问题1：要沿着特定道路行走吗？
问题2：给shelter和neighorhood进行一样的正规化处理。
问题3：确定空间范围的获取和target的选择。


## CREATE A MODEL

In [2]:
import importlib.util
# Import pyflamegpu and some other libraries we will use in the tutorial
import pyflamegpu
import sys, random, math
import matplotlib.pyplot as plt

In [3]:
%env CUDA_PATH=D:\cuda12.9

env: CUDA_PATH=D:\cuda12.9


In [4]:
def create_model():
    model = pyflamegpu.ModelDescription("so-phy-shelter")
    return model

## Agent 相关的东西

| Model Variable | Agent Variable Name | Description
| :--- | :--- | :--- |
| $AIE$ | `Around_in_evacuation` | agent周围在疏散的人 |
| $m$ | `metabolism` | Metabolism |
| $w$ | `sugar_level` | Sugar Wealth |
|  | `env_sugar_max` | Each cell has a maximum sugar level which can be no greater than $S_{max}$

In [6]:
student_agent=model.newAgent("student_agent")

student_agent.newstate("focused")
student_agent.newstate("not evacuate")
student_agent.newstate("building evacuate")
student_agent.newstate("stairwell evacuate")
student_agent.newstate("neighborhood evacuate")

student_agent.newVariableInt("x")
student_agent.newVariableInt("y")
student_agent.newVariableInt("agent_id")
student_agent.newVariableInt("target_stairwell_id")
student_agent.newVariableInt("target_shelter_id")



NameError: name 'model' is not defined

In [ ]:
def define_agents(model):
    """
        student agent
    """

    #create the agent
    agent = model.newAgent("student_agent")

    # Assign its variables
    agent.newVariableFloat("x")
    agent.newVariableFloat("y")
    agent.newVariableFloat("vx")
    agent.newVariableFloat("vy")
    agent.newVariableFloat("steer_x")
    agent.newVariableFloat("steer_y")
    agent.newVariableInt("agent_id")
    agent.newVariableInt("target_stairwell_id")
    agent.newVariableInt("target_shelter_id")

    # Assign its functions
    find_stairwell_fn = agent.newRTCFunction("find_stairwell", pyflamegpu.codegen.translate(find_stairwell))
    find_shelter_fn = agent.newRTCFunction("find_shelter", pyflamegpu.codegen.translate(find_shelter))
    walk_move_fn = agent.newRTCFunction("walk_move", pyflamegpu.codegen.translate(walk_move))
    downstair_move_fn = agent.newRTCFunction("downstair_move", pyflamegpu.codegen.translate(downstair_move))

    """
        stairwell agent
    """   

    #create the agent
    agent = model.newAgent("stairwell_agent")

### agent的状态变更


```python
m = pyflamegpu.ModelDescription("model")
# It contains an agent with 'variable 'x' and two states 'foo' and 'bar'
a = m.newAgent("agent")
a.newVariableInt("x")
a.newState("foo")
a.newState("bar")
```

如何设置只有处于focus或者evacuate状态的agent才能执行某种函数？
- 答案是：为执行函数设置进入函数和出去函数。
```python
af1 = a.newRTCFunction("example_function", ExampleFn_source)
af1.setInitialState("foo")
```


如何设置状态变更的函数？例如：设置agent没到楼梯口不会下楼
```python
#条件函数定义 (Python)
@pyflamegpu.agent_function_condition
def py_x_is_1() -> bool:
    return pyflamegpu.getVariableInt("x") == 1

#条件函数绑定
af1 = a.newRTCFunction("example_function", ExampleFn_source)
af1.setInitialState("foo")
af1.setEndState("bar")
x_is_1_translated = pyflamegpu.codegen.translate(x_is_1)
af1.setRTCFunctionCondition(x_is_1_translated)
```

这里可以设置为building_move函数。
函数的内容就包括进行基本移动以及判断是否移动到了那个点。

## Define Environmental Properties

In [3]:
def define_environment(model):
    """
        Environment
    """
    env = model.Environment()

    #activate_radium
    env.newPropertyFloat("Range_be_activated", 15.0)
    env.newPropertyFloat("Range_find_stairwell", 100.0)
    env.newPropertyFloat("Range_find_shelter", 500.0)
    env.newPropertyFloat("threshold_stairwell_congestion", 10.0)
    
    

# Define Messages

In [ ]:
def define_messages(model):
    """
      Location messages
    """  
    message = model.newMessageBruteForce("student_agent_location_message")
    message.newVariableID("id")
    message.newVariableFloat("x")
    message.newVariableFloat("y")

    message = model.newMessageBruteForce("shelter_agent_location_message")
    message.newVariableID("id")
    message.newVariableFloat("x")
    message.newVariableFloat("y")

    message = model.newMessageBruteForce("stairwell_agent_location_message")
    message.newVariableID("id")
    message.newVariableFloat("x")
    message.newVariableFloat("y")


    """
      stairwell state messages 
    """
    message = model.newMessageBruteForce("stairwell_congestion_message")
    message.newVariableID("id")
    message.newVariableFloat("congestion_state")

    """
      student agent state messages
    """
    message = model.newMessageBruteForce("student_evacuate_state_message")
    message.newVariableID("id")
    message.newVariableFloat("evacuate state")



我现在的问题呢，就是：message获取的范围，我要用message本身的好还是用函数遍历的好。我去查一查手册

ok。我现在懂了：
用message本身返回的应该是：交互半径中小格子的message，所以可能获取的范围会更大。但是我认为是可以接受的。

    for message in message_in(x1, y1):
        if message.getVariableUInt("id") != ID :
            x2 = message.getVariableFloat("x")
            y2 = message.getVariableFloat("y") 
            x21 = x2 - x1
            y21 = y2 - y1
            separation = math.sqrtf(x21*x21 + y21*y21)
            if separation < RADIUS and separation > 0 :

## functions

## Function 的顺序很重要

但是我还有疑问：

不同的agent的函数能够并发吗？

不同agent的函数怎么绑定？

首先第一件事就是做一个简单的移动函数。

## 初始化population

我现在集齐了碎片：population的地理位置，状态，速度，熟悉度的差异。

熟悉度可以参考设置不同的策略

也可以设置各种问卷：例如你们会不会倾向于用手机联系同学。

小trick：
参考博弈论的code部分
- 可以为evacuees 的疏散状态设置为特征值
- 可以用随机的方法随机一个，然后添加到状态里面
~~~python
energy = max(
    random.normalvariate(INIT_ENERGY_MU, INIT_ENERGY_SIGMA), INIT_ENERGY_MIN
)
if MAX_ENERGY > 0.0:
    energy = min(energy, MAX_ENERGY)
instance.setVariableFloat("energy", energy)
~~~


困难是设置两种不同的agent

第一种方法：

先把create_agents函数嵌入到cudaSimulation里面去，再实例化。

cudaSimulation = pyflamegpu.CUDASimulation(model)

In [ ]:
class create_agents(pyflamegpu.HostFunction):
    def run(self, FLAMEGPU):
        Student_AGENT_COUNT = FLAMEGPU.environment.getPropertyUInt("Student_AGENT_COUNT") #总共有多少个学生agent

model.addInitFunction(create_agents())     

cudaSimulation = pyflamegpu.CUDASimulation(model)

NameError: name 'model' is not defined

第二种方法：
先实例化，再用setPopulationData的操作把agent的实际信息加入进去

In [ ]:
cudaSimulation = pyflamegpu.CUDASimulation(model)

random.seed(cudaSimulation.SimulationConfig().random_seed)

studentPopulation = pyflamegpu.AgentVector(model.Agent("prey"), num_student_agent)
for i in range(0, num_student_agent):
    Student_AGENT = studentPopulation[i]
    Student_AGENT.setVariableFloat("x", random.uniform(-1.0, 1.0))
    Student_AGENT.setVariableFloat("y", random.uniform(-1.0, 1.0))
    prey.setVariableFloat("vx", random.uniform(-1.0, 1.0))
    prey.setVariableFloat("vy", random.uniform(-1.0, 1.0))
    prey.setVariableFloat("steer_x", 0.0)
    prey.setVariableFloat("steer_y", 0.0) 
    prey.setVariableFloat("type", 1.0)
    prey.setVariableInt("life", random.randint(0, 50))

cudaSimulation.setPopulationData(predatorPopulation)

如何设置agent的行为逻辑，包含心理和社会要素

 PATH-U 的路径寻找（wayfinding）智能体（Qi Yang 2025）最新的一个wayfinding论文

 智能体根据目的地（AdestAdest​）、空间能力（AsbscolAsbscol​）、空间知识（AspkAspk​）和起点（AoriginAorigin​）生成，并从起点随机选择一个方向开始移动。

 检查是否到达目的地： 如果目的地可见（在视野列表 L f o v L fov ​ 中），智能体直接移动至目的地并终止。 
 
 探索阶段： 如果目的地不可见，智能体进入探索模式（explore），尝试通过空间知识或随机探索找到路径。 
 
 楼层判断： 智能体检查当前楼层（floorcurrent）是否与目标楼层（floortest）一致。 
 
 如果楼层不同，调用 FloorStrategy 处理跨楼层路径（例如使用电梯或楼梯）。 
 
 不确定性计算与局部策略： 计算当前路径的不确定性（ U w f U wf ​ ），基于当前位置、目的地、视野、记忆和空间能力。 
 
 调用 LocalStrategy（分为 L1 和 L2 模式）选择下一步移动的节点（ N s u b N o d e N subNode ​ ）。
 
  L1：直接选择可见的路径节点。 L2：从可用路径列表（ R R）中选择最优节点。 决策点处理： 如果当前位置是决策点（ N d p N dp ​ 中的节点），检查是否有指向目的地的标识（helpfulSign）。若有，直接移动至目标节点并更新记忆（ M M）。
  
   移动与记忆更新： 移动到子节点（ N s u b N o d e N subNode ​ ），并更新短期记忆缓冲区（ M M）以记录已访问的节点。 终止条件： 当智能体到达目的地（ A p o s = A d e s t A pos ​ =A dest ​ ）时，过程结束。

但是上述的path-u 基本没有考虑到社区内熟悉度，也没有考虑到拥挤和前方人员的影响。

如果要考虑到避免碰撞和实现寻路算法的话，打格子还是必要的，怎么将网格嵌入到3d的空间里面也很重要。